# Running headcam with top-down Basler cam

In [1]:
from multicamera_acquisition.acquisition import refactor_acquire_video, reset_loggers
from multicamera_acquisition.config import (
    load_config,
)
import datetime
import logging
from os.path import join, exists
import pandas as pd
from glob import glob
import matplotlib.pyplot as plt
import numpy as np

/Users/jonahpearl/Documents/PiN/Datta_lab/Local_code/multicamera_acquisition/multicamera_acquisition/interfaces/camera_azure.py:18: UserWarning: pyk4a not installed.  Azure cameras will not be available.
  warnings.warn("pyk4a not installed.  Azure cameras will not be available.")


In [2]:
# Recording params
rec_time_s = 10
# rec_time_s = 15 * 60
final_writer_timeout = 1
# base_path = "/Users/brianxu/Documents/face_cam/data"
base_path = "/Users/jonahpearl/Documents/PiN/Datta_lab/Experiments/20241111_facecam_testing/"
config = load_config("./facecam_top_config.yaml")
mouse = "W00102"
config

{'acq_loop': {'display_every_n': 1,
  'downsample': 4,
  'dropped_frame_warnings': True,
  'max_frames_to_acqure': None},
 'cameras': {'face_cam': {'brand': 'uvc',
   'brightness': 32,
   'restart_after_n_frame_lost': 50,
   'contrast': 100,
   'display': {'display_frames': True, 'display_range': (0, 255)},
   'exposure': 50,
   'exposure_mode': 1,
   'exposure_priority': 0,
   'fps': 210,
   'frame_size': (640, 400),
   'gain': 50,
   'gamma': 340,
   'id': 0,
   'name': 'face_cam',
   'roi': None,
   'trigger': {'trigger_type': 'no_trigger'},
   'writer': {'auto_remux_videos': True,
    'camera_name': 'face_cam',
    'codec': 'h264',
    'constqp': 18,
    'depth': False,
    'fmt': 'YUV420',
    'fps': 210,
    'gop': '30',
    'gpu': None,
    'idrperiod': '256',
    'loglevel': 'info',
    'max_video_frames': None,
    'multipass': '0',
    'output_px_format': 'yuv420p',
    'pixel_format': 'gray8',
    'preset': 'ultrafast',
    'profile': 'high',
    'quality': 15,
    'rc': 'co

In [3]:
config["cameras"].pop("top")

{'brand': 'basler',
 'constqp': 10,
 'display': {'display_frames': True, 'display_range': (0, 255)},
 'exposure': 1000,
 'fps': 120,
 'gain': 12,
 'gamma': 1,
 'id': '24510726',
 'name': 'top',
 'pixel_format': 'Mono8',
 'roi': None,
 'trigger': {'trigger_type': 'no_trigger'},
 'writer': {'auto_remux_videos': True,
  'camera_name': 'top',
  'codec': 'h264',
  'constqp': 18,
  'depth': False,
  'fmt': 'YUV420',
  'fps': 120,
  'gop': '30',
  'gpu': None,
  'idrperiod': '256',
  'loglevel': 'error',
  'max_video_frames': 10368000,
  'multipass': '0',
  'output_px_format': 'yuv420p',
  'pixel_format': 'gray8',
  'preset': 'P1',
  'profile': 'high',
  'quality': 15,
  'rc': 'constqp',
  'tuning_info': 'ultra_low_latency',
  'type': 'ffmpeg',
  'video_codec': 'libx264'}}

In [4]:
fps = 210
config["cameras"]["face_cam"]["display"]["display_frames"] = True
config["cameras"]["face_cam"]["restart_after_n_frame_lost"] = 15
config["cameras"]["face_cam"]["brightness"] = 64
config["cameras"]["face_cam"]["fps"] = fps
config["cameras"]["face_cam"]["writer"]["gpu"] = None
config["cameras"]["face_cam"]["writer"]["loglevel"] = "info"
config["cameras"]["face_cam"]["writer"]["fps"] = fps
config["rt_display"]["downsample"] = 1

config["acq_loop"]['dropped_frame_warnings'] = True
# config["acq_loop"]['max_frames_to_acqure'] = 190 * rec_time_s

In [5]:
datestr = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
recording_name = f"{datestr}_{mouse}"
recording_name

'2024-11-11_16-52-58_W00102'

In [6]:
reset_loggers()

save_loc, full_config = refactor_acquire_video(
    join(base_path, mouse),
    config,
    recording_duration_s=rec_time_s,
    recording_name=recording_name,
    # recording_name="tmp",
    append_datetime=False,
    final_writer_timeout=final_writer_timeout,
    overwrite=True,
    logging_level=logging.CRITICAL,
)

/Users/jonahpearl/Documents/PiN/Datta_lab/Local_code/multicamera_acquisition/multicamera_acquisition/interfaces/camera_azure.py:18: UserWarning: pyk4a not installed.  Azure cameras will not be available.
  warnings.warn("pyk4a not installed.  Azure cameras will not be available.")
Process face_cam_acqLoop:
Traceback (most recent call last):
  File "/Users/jonahpearl/Documents/PiN/Datta_lab/Local_code/multicamera_acquisition/multicamera_acquisition/interfaces/camera_uvc.py", line 205, in _create_uvc_cam
    self.cam = self.system.Capture(devices[self.device_index]['uid'])
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "uvc_bindings.pyx", line 536, in uvc_bindings.Capture.__init__
  File "uvc_bindings.pyx", line 592, in uvc_bindings.Capture._init_device
uvc_bindings.OpenError: Could not open device. Error: Access denied

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/miniconda3/envs/dataPy

CameraError: Acq loop for face_cam failed to initialize.

In [ ]:
for cam in config["cameras"]:
    metadata_file = glob(join(save_loc, f"*{cam}*metadata.csv"))[0]
    df = pd.read_csv(metadata_file, header=0)
    diffs = np.diff(df.frame_timestamp)
    plt.plot(diffs[1:])
    break

In [ ]:
df.head()